In [ ]:
#-------------------------------------------------------------------------------
# Name:        01_check_bbox
# Purpose:     Rasterize the vecor input mask so that they are 256x256px
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/19wF5gdNnmX4Lwj1kHxltKGRF6DT1yvQ7

# Vector Masks:
# Output from the selection of IA areas in a 2560x2560 bounding box

# Raster Masks:
# Input for the modelling, as a direct conversion from vector to raster

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

/Users/misanmeggison/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


2
EPSG:4326


,id,url,image_date,min_date,max_date,geometry
0,0,https://apps.sentinel-hub.com/eo-browser/?zoom...,2024-03-14,2024-03-13,2024-03-15,"MULTIPOLYGON (((18.67097 -4.56564, 18.67772 -4..."
1,1,https://apps.sentinel-hub.com/eo-browser/?zoom...,2024-03-14,2024-03-13,2024-03-15,"MULTIPOLYGON (((18.69332 -4.55245, 18.69921 -4..."


# Read the Image files from Drive

In [ ]:
import geopandas as gpd
import os

# path to the input data
# parent_dir = '/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_IA'
parent_dir = os.path.dirname(os.getcwd())
aoi_path = os.path.join(parent_dir, 'Iter4_49_IA.geojson')

gdf_ISL_lines = gpd.read_file(aoi_path)

print(len(gdf_ISL_lines)) # total lines in the AoI
print(gdf_ISL_lines.crs)  # current projection
gdf_ISL_lines.head(3)

# Create Geojson of Aoi Block from the Labels

In [ ]:
from shapely.geometry import Polygon
import shapely.geometry as geometry
import os

resent_AoI = []
resent_AoI_small = []
selected_AoI = []

aoi_id = 'IA_49' # specify aoi

# Modified attr_dict to include new attributes
attr_dict = {
    'geometry': [],
    'image_AoI': [],
    'min_date': [],
    'max_date': [],
    'url': [],
    'image_date': []
}


filtered_df = gdf_ISL_lines

filtered_df.total_bounds

limLngMin=filtered_df.total_bounds[0] #["minx"]
limLatMin=filtered_df.total_bounds[1] #["miny"]
limLngMax=filtered_df.total_bounds[2] #["maxx"]
limLatMax=filtered_df.total_bounds[3] #["maxy"]

#set the bounds:
bound_points_shapely = geometry.MultiPoint([(limLngMin, limLatMin), (limLngMax, limLatMax)])

# get lat/lng from center (below: True centroid - since coords may be multipoint)
input_lon_center = bound_points_shapely.centroid.coords[0][0]
input_lat_center = bound_points_shapely.centroid.coords[0][1]

# set the epsg_code
utm_code=32634
    # define project_from_to with iteration
    # see https://gis.stackexchange.com/a/127432/33092

    # get the bounds
lon_lat_list=((limLngMin,limLatMin ),(limLngMin,limLatMax),(limLngMax,limLatMax),(limLngMax,limLatMin))

polygon_geom = Polygon(lon_lat_list)
polygon = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[polygon_geom]) #careful with the hard coded EPSG

    # Reproject to a system with meters:
polygon_rp = polygon.to_crs(epsg=utm_code)

polygon_width = polygon_rp.bounds.maxx - polygon_rp.bounds.minx
    #print("W: ", polygon_width)

polygon_height = polygon_rp.bounds.maxy - polygon_rp.bounds.miny
    #print("H: ", polygon_height)

    ##
    ### the area should have a minimum of 9 patches meaning extent should be
    ### [3 x 128 x 10 = 3840m] by [3 x 128 x 10 = 3840m]
    ##

f=polygon_rp.centroid
    #print("f is",f[0].coords[0])
centre_coords=f[0].coords[0]

    # creating an empty geometry:
geo_new_polygon=[]

boo_width = False
boo_height = False

    # set the new cell dimentions:
if polygon_width[0]//2560 + 1 < 3:
    cells_w = 3
    boo_width = True
else:
    cells_w = polygon_width[0]//2560 + 1

if polygon_height[0]//2560 + 1 < 3:
    cells_h = 3
    boo_height = True
else:
    cells_h = polygon_height[0]//2560 + 1

    #Set the minimal criteria:
    # set the new cell dimentions:
boo_width2 = True
boo_height2 = True
if polygon_width[0]//2560 >1:
    boo_width2 = False

if polygon_height[0]//2560>1:
    boo_height2 = False

    # correction factor for all directions
factor = ((256 / 2) * 10)

    # adding a buffer of one cell to all directions:
newx1=centre_coords[0] - (factor * cells_w + 10)
newx2=centre_coords[0] + (factor * cells_w + 10)

newy1=centre_coords[1] - (factor * cells_h + 10)
newy2=centre_coords[1] + (factor * cells_h + 10)

new_xy_coords=((newx1,newy1),(newx2,newy1),(newx2,newy2),(newx1,newy2))
new_polygon=Polygon(new_xy_coords)

# The new geometry:
geo_new_polygon.append(new_polygon)

geo_new_polygon


attr_dict['geometry'].append(geo_new_polygon[0])
attr_dict['image_AoI'].append(aoi_id)

# Example appending attributes from the first row of filtered_df,
# you might need to adjust this based on your actual logic for multiple polygons.
attr_dict['min_date'].append(filtered_df['min_date'].iloc[0])
attr_dict['max_date'].append(filtered_df['max_date'].iloc[0])
attr_dict['url'].append(filtered_df['url'].iloc[0])
attr_dict['image_date'].append(filtered_df['image_date'].iloc[0])


    # The new data frame:
polygon_new = gpd.GeoDataFrame(attr_dict,crs="epsg:{}".format(utm_code))

polygon_to_qc=polygon_new.to_crs("epsg:4326")

output_file = aoi_id + ".geojson"
output_file = os.path.join(parent_dir, output_file)
polygon_to_qc.to_file(output_file)

if boo_width and boo_height:
    resent_AoI.append(output_file)
else:
    selected_AoI.append(output_file)

# if boo_width2 and boo_height2:
#     resent_AoI_small.append(output_file)

polygon_to_qc

,geometry,image_AoI,min_date,max_date,url,image_date
0,"POLYGON ((18.65303 -4.59464, 18.72239 -4.59487...",IA_49,2024-03-13,2024-03-15,https://apps.sentinel-hub.com/eo-browser/?zoom...,2024-03-14
